### 1. Initialization and read


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, date_sub, lead


In [0]:
df = spark.table("catalogue_project1.bronze.erp_px_cat_g1v2")

In [0]:
%sql
SELECT * FROM catalogue_project1.bronze.erp_px_cat_g1v2 limit 10
-- select distinct MAINTENANCE from catalogue_project1.bronze.erp_px_cat_g1v2

### 2. Transformations
2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


2.2 Normalize maintenance

In [0]:
df = df.withColumn(
    "maintenance", 
    F.when(F.upper(col("maintenance")).isin("Y", "YES"), "Yes")
     .when(F.upper(col("maintenance")).isin("N", "NO"), "No")
     .otherwise("n/a")
)


2.3 Rename columns

In [0]:
RENAME_CONFIG = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.erp_px_cat_g1v2")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.erp_px_cat_g1v2 LIMIT 10
